# 02 - Autograd and the Computation Graph

## Learning Objectives
- Understand how `requires_grad` enables automatic differentiation
- Visualize the computational graph (nodes = operations, edges = data flow)
- Use `.backward()` to compute gradients and `.grad` to access them
- Understand when and why to use `detach()` and `torch.no_grad()`
- Avoid the gradient accumulation pitfall with `zero_grad()`
- Verify autograd results against manual gradient calculations

## Prerequisites
- Notebook 01 (Tensors, Shapes, Dtypes & Broadcasting)
- Basic calculus (derivatives, chain rule)
- PyTorch installed (`pip install torch`)

## Table of Contents
1. [Why Autograd?](#1-why-autograd)
2. [requires_grad: Tracking Computation](#2-requires_grad-tracking-computation)
3. [The Computational Graph](#3-the-computational-graph)
4. [.backward() and .grad](#4-backward-and-grad)
5. [detach() and torch.no_grad()](#5-detach-and-torchno_grad)
6. [Gradient Accumulation and zero_grad()](#6-gradient-accumulation-and-zero_grad)
7. [Manual Gradient vs Autograd Comparison](#7-manual-gradient-vs-autograd-comparison)
8. [Computation Graph Concepts](#8-computation-graph-concepts)
9. [Exercises](#9-exercises)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)

In [ ]:
import torch
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

## 1. Why Autograd?

Training a neural network requires computing gradients of the **loss** with respect to all **parameters**:

$$
\theta_{t+1} = \theta_t - \eta \frac{\partial \mathcal{L}}{\partial \theta_t}
$$

For millions of parameters and complex architectures, computing these gradients by hand is infeasible. PyTorch's **autograd** engine does it automatically using **reverse-mode automatic differentiation** (backpropagation).

Key idea: PyTorch records every operation on tensors that have `requires_grad=True` into a **dynamic computational graph**, then traverses this graph backwards to compute all gradients in one pass.

## 2. requires_grad: Tracking Computation

Setting `requires_grad=True` tells PyTorch: "I want gradients for this tensor."

In [ ]:
# By default, requires_grad=False
a = torch.tensor([2.0, 3.0])
print(f"a.requires_grad = {a.requires_grad}")

# Enable gradient tracking
x = torch.tensor([2.0, 3.0], requires_grad=True)
print(f"x.requires_grad = {x.requires_grad}")

# Operations on tracked tensors produce tracked results
y = x * 2 + 1
print(f"y.requires_grad = {y.requires_grad}")
print(f"y.grad_fn = {y.grad_fn}")  # shows the operation that created y

In [ ]:
# You can also enable tracking after creation
t = torch.randn(3)
print(f"Before: requires_grad={t.requires_grad}")

t.requires_grad_(True)  # in-place toggle
print(f"After:  requires_grad={t.requires_grad}")

In [ ]:
# Leaf vs non-leaf tensors
x = torch.tensor(3.0, requires_grad=True)  # leaf (created by user)
y = x ** 2                                   # non-leaf (created by operation)
z = 2 * y + 1                                # non-leaf

print(f"x is leaf: {x.is_leaf}, grad_fn: {x.grad_fn}")
print(f"y is leaf: {y.is_leaf}, grad_fn: {y.grad_fn}")
print(f"z is leaf: {z.is_leaf}, grad_fn: {z.grad_fn}")

## 3. The Computational Graph

When you perform operations on tensors with `requires_grad=True`, PyTorch builds a **directed acyclic graph (DAG)**:

- **Nodes** represent operations (recorded in `grad_fn`)
- **Edges** represent data flow (tensor -> operation -> tensor)
- **Leaves** are input tensors (parameters)
- **Root** is the output (usually the loss)

```
Example: z = 2 * (x^2) + 1

    x (leaf)
    |
   [PowBackward0]   (x^2)
    |
    y
    |
   [MulBackward0]   (2 * y)
    |
   [AddBackward0]   (+ 1)
    |
    z (root)
```

The graph is **dynamic** -- it is rebuilt on every forward pass, enabling Python control flow (if/else, loops).

In [ ]:
# Tracing the computation graph via grad_fn
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
z = 2 * y + 1

# Walk backwards through the graph
print(f"z.grad_fn: {z.grad_fn}")
print(f"  -> next: {z.grad_fn.next_functions}")
print(f"    -> next: {z.grad_fn.next_functions[0][0].next_functions}")
print(f"      -> next: {z.grad_fn.next_functions[0][0].next_functions[0][0].next_functions}")

## 4. .backward() and .grad

`.backward()` triggers backpropagation from a scalar output, computing $\frac{\partial \text{output}}{\partial \text{leaf}}$ for every leaf tensor with `requires_grad=True`.

In [ ]:
# Simple example: z = 2*x^2 + 3*x + 1, at x = 2.0
# dz/dx = 4*x + 3 = 4*2 + 3 = 11

x = torch.tensor(2.0, requires_grad=True)
z = 2 * x**2 + 3 * x + 1

print(f"z = {z.item()}")
print(f"x.grad before backward: {x.grad}")

z.backward()

print(f"x.grad after backward:  {x.grad}")
print(f"Expected: 4*2 + 3 = {4*2 + 3}")

In [ ]:
# Multiple inputs
# f(a, b) = a^2 * b + b^3
# df/da = 2ab, df/db = a^2 + 3b^2

a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

f = a**2 * b + b**3
f.backward()

print(f"f = {f.item()}")
print(f"df/da = {a.grad.item()} (expected: 2*2*3 = {2*2*3})")
print(f"df/db = {b.grad.item()} (expected: 4 + 27 = {4 + 27})")

In [ ]:
# Gradients for vector-valued inputs
# backward() requires a scalar output. For vector outputs, use .sum() or provide gradient argument.

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2  # element-wise: [1, 4, 9]

# Option 1: sum first, then backward
y.sum().backward()
print(f"dy_sum/dx = {x.grad}")  # [2, 4, 6] = 2*x

# Reset
x.grad.zero_()

# Option 2: provide gradient argument (Jacobian-vector product)
y = x ** 2
v = torch.ones_like(y)  # "upstream gradient"
y.backward(gradient=v)
print(f"dy/dx (with gradient=ones) = {x.grad}")  # same result

## 5. detach() and torch.no_grad()

Sometimes you want to **stop gradient tracking**:
- **Inference** (evaluation): no need to compute gradients
- **Freezing layers**: prevent gradient flow to certain parameters
- **Extracting values**: get a tensor value without the computation graph

In [ ]:
# detach(): creates a new tensor detached from the computation graph
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

y_detached = y.detach()  # same data, but no gradient tracking
print(f"y.requires_grad:          {y.requires_grad}")
print(f"y_detached.requires_grad: {y_detached.requires_grad}")
print(f"y_detached.grad_fn:       {y_detached.grad_fn}")

# Shares memory with y!
print(f"Same data: {y_detached.data_ptr() == y.data_ptr()}")

In [ ]:
# torch.no_grad(): context manager that disables gradient tracking
# More efficient than detach() for entire blocks of code

x = torch.tensor(3.0, requires_grad=True)

# With gradient tracking (normal)
y1 = x * 2
print(f"With grad:    y1.requires_grad={y1.requires_grad}, grad_fn={y1.grad_fn}")

# Without gradient tracking
with torch.no_grad():
    y2 = x * 2
    print(f"No grad:      y2.requires_grad={y2.requires_grad}, grad_fn={y2.grad_fn}")

In [ ]:
# Practical use: evaluation mode
# During inference, we don't need gradients => faster and less memory

model_weight = torch.randn(3, 3, requires_grad=True)

# Simulated inference
with torch.no_grad():
    input_data = torch.randn(3)
    output = model_weight @ input_data
    print(f"Output: {output}")
    print(f"Output requires_grad: {output.requires_grad}")  # False - good!

In [ ]:
# Use case: updating parameters without tracking
# (This is what optimizers do internally)

param = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss = (param ** 2).sum()
loss.backward()

print(f"Before update: param = {param.data}")
print(f"Gradient:      param.grad = {param.grad}")

# Update without tracking (use .data or no_grad)
with torch.no_grad():
    param -= 0.1 * param.grad  # gradient descent step

print(f"After update:  param = {param.data}")

## 6. Gradient Accumulation and zero_grad()

**Critical pitfall**: PyTorch **accumulates** gradients by default. Calling `.backward()` multiple times will **add** to existing `.grad` values rather than replacing them.

In [ ]:
# Demonstration of gradient accumulation
x = torch.tensor(3.0, requires_grad=True)

# First backward pass
y1 = x ** 2  # dy1/dx = 2*x = 6
y1.backward()
print(f"After 1st backward: x.grad = {x.grad}")

# Second backward pass (WITHOUT zeroing gradients)
y2 = x ** 2
y2.backward()
print(f"After 2nd backward: x.grad = {x.grad}  (accumulated! 6 + 6 = 12)")

# Third backward pass (WITHOUT zeroing gradients)
y3 = x ** 2
y3.backward()
print(f"After 3rd backward: x.grad = {x.grad}  (accumulated! 12 + 6 = 18)")

In [ ]:
# The fix: zero_grad() before each backward pass
x = torch.tensor(3.0, requires_grad=True)

for i in range(3):
    if x.grad is not None:
        x.grad.zero_()     # Reset gradients!
    y = x ** 2
    y.backward()
    print(f"Iteration {i}: x.grad = {x.grad}")

In [ ]:
# Why accumulation exists: it's actually useful for gradient accumulation
# when you can't fit a full batch in memory

# Simulated micro-batching
weight = torch.tensor(2.0, requires_grad=True)
micro_batches = [torch.tensor(1.0), torch.tensor(2.0), torch.tensor(3.0)]

# Accumulate gradients across micro-batches
for mb in micro_batches:
    loss = (weight * mb - mb**2) ** 2  # some loss
    loss.backward()  # gradients accumulate

print(f"Accumulated gradient: {weight.grad}")

# Then do one update
with torch.no_grad():
    weight -= 0.01 * weight.grad
    weight.grad.zero_()  # reset for next iteration

## 7. Manual Gradient vs Autograd Comparison

Let's verify that autograd gives us the correct gradients by comparing with hand-computed derivatives.

In [ ]:
# Example 1: Simple polynomial
# f(x) = 3x^3 - 2x^2 + x - 5
# f'(x) = 9x^2 - 4x + 1

x_val = 2.0
x = torch.tensor(x_val, requires_grad=True)

f = 3 * x**3 - 2 * x**2 + x - 5
f.backward()

manual_grad = 9 * x_val**2 - 4 * x_val + 1
print(f"f(x)  = {f.item()}")
print(f"Autograd: df/dx = {x.grad.item()}")
print(f"Manual:   df/dx = {manual_grad}")
print(f"Match: {abs(x.grad.item() - manual_grad) < 1e-6}")

In [ ]:
# Example 2: Chain rule
# f(x) = sin(x^2)
# f'(x) = cos(x^2) * 2x  (chain rule)

x_val = 1.5
x = torch.tensor(x_val, requires_grad=True)

f = torch.sin(x ** 2)
f.backward()

manual_grad = np.cos(x_val**2) * 2 * x_val
print(f"f(x)  = {f.item():.6f}")
print(f"Autograd: df/dx = {x.grad.item():.6f}")
print(f"Manual:   df/dx = {manual_grad:.6f}")
print(f"Match: {abs(x.grad.item() - manual_grad) < 1e-6}")

In [ ]:
# Example 3: Multi-variable with chain rule
# L = (w*x + b - y)^2   (MSE loss for one sample)
# dL/dw = 2*(w*x + b - y) * x
# dL/db = 2*(w*x + b - y)

w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)
x_val = 2.0
y_val = 3.0
x = torch.tensor(x_val)
y = torch.tensor(y_val)

# Forward pass
pred = w * x + b
loss = (pred - y) ** 2

# Backward pass
loss.backward()

# Manual computation
residual = 0.5 * x_val + 0.1 - y_val  # w*x + b - y = -1.9
manual_dw = 2 * residual * x_val
manual_db = 2 * residual

print(f"Loss = {loss.item():.4f}")
print(f"\ndL/dw: autograd={w.grad.item():.4f}, manual={manual_dw:.4f}")
print(f"dL/db: autograd={b.grad.item():.4f}, manual={manual_db:.4f}")

In [ ]:
# Example 4: Vector-valued function
# f(x) = ||x||^2 = sum(x_i^2)
# df/dx_i = 2*x_i

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
f = (x ** 2).sum()  # = 1 + 4 + 9 = 14
f.backward()

manual_grad = 2 * torch.tensor([1.0, 2.0, 3.0])
print(f"f(x) = {f.item()}")
print(f"Autograd: {x.grad}")
print(f"Manual:   {manual_grad}")
print(f"Match: {torch.allclose(x.grad, manual_grad)}")

## 8. Computation Graph Concepts

### Dynamic vs Static Graphs
PyTorch uses **dynamic** (define-by-run) graphs, rebuilt each forward pass. This means:
- Python control flow (if/else, loops) works naturally
- The graph can change every iteration
- Easy to debug with standard Python tools

In [ ]:
# Dynamic graph: different paths each forward pass
def dynamic_function(x):
    """The computation graph changes based on x's value."""
    y = x ** 2
    if x.item() > 0:
        return y * 2    # graph: x -> pow -> mul
    else:
        return y + 1    # graph: x -> pow -> add

# Positive input
x_pos = torch.tensor(3.0, requires_grad=True)
out = dynamic_function(x_pos)
out.backward()
print(f"x=3.0: f(x)={out.item()}, grad={x_pos.grad.item()} (expected: 4*3=12)")

# Negative input
x_neg = torch.tensor(-2.0, requires_grad=True)
out = dynamic_function(x_neg)
out.backward()
print(f"x=-2.0: f(x)={out.item()}, grad={x_neg.grad.item()} (expected: 2*(-2)=-4)")

In [ ]:
# The graph is freed after backward() by default
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()

# Cannot call backward() again on the same graph
try:
    y.backward()
except RuntimeError as e:
    print(f"Error: {e}")

# To retain the graph, use retain_graph=True
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward(retain_graph=True)
print(f"First backward:  x.grad = {x.grad}")

x.grad.zero_()
y.backward()  # works now!
print(f"Second backward: x.grad = {x.grad}")

In [ ]:
# Higher-order gradients (gradient of gradient)
# f(x) = x^3, f'(x) = 3x^2, f''(x) = 6x

x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

# First derivative
grad1 = torch.autograd.grad(y, x, create_graph=True)[0]
print(f"f'(x) = {grad1.item()} (expected: 3*4 = 12)")

# Second derivative
grad2 = torch.autograd.grad(grad1, x)[0]
print(f"f''(x) = {grad2.item()} (expected: 6*2 = 12)")

In [ ]:
# Visualizing gradient flow in a simple neural network forward pass
torch.manual_seed(42)

# Input
x = torch.randn(1, 3)

# Layer 1: Linear + ReLU
W1 = torch.randn(3, 4, requires_grad=True)
b1 = torch.randn(4, requires_grad=True)

# Layer 2: Linear
W2 = torch.randn(4, 1, requires_grad=True)
b2 = torch.randn(1, requires_grad=True)

# Forward pass
z1 = x @ W1 + b1           # linear
a1 = torch.relu(z1)        # activation
z2 = a1 @ W2 + b2          # linear
loss = z2.squeeze() ** 2   # simple scalar loss

print("Computation graph trace:")
print(f"  loss.grad_fn: {loss.grad_fn}")
print(f"  -> z2.grad_fn: {z2.grad_fn}")
print(f"  -> a1.grad_fn: {a1.grad_fn}")
print(f"  -> z1.grad_fn: {z1.grad_fn}")

# Backward pass
loss.backward()

print(f"\nGradients computed:")
print(f"  W1.grad shape: {W1.grad.shape}")
print(f"  b1.grad shape: {b1.grad.shape}")
print(f"  W2.grad shape: {W2.grad.shape}")
print(f"  b2.grad shape: {b2.grad.shape}")
print(f"  x.grad: {x.grad} (None because requires_grad=False)")

## 9. Exercises

### Exercise 1: Compute Gradients for a Multi-Variable Function

Given the function:

$$f(x, y, z) = x^2 y + yz^3 - 2xz$$

Compute $\frac{\partial f}{\partial x}$, $\frac{\partial f}{\partial y}$, and $\frac{\partial f}{\partial z}$ at the point $(x, y, z) = (1, 2, 3)$.

Verify against the analytical gradients:
- $\frac{\partial f}{\partial x} = 2xy - 2z$
- $\frac{\partial f}{\partial y} = x^2 + z^3$
- $\frac{\partial f}{\partial z} = 3yz^2 - 2x$

In [ ]:
# TODO: Compute gradients using autograd and verify against manual computation

# x = torch.tensor(1.0, requires_grad=True)
# y = torch.tensor(2.0, requires_grad=True)
# z = torch.tensor(3.0, requires_grad=True)

# f = ...
# f.backward()

# print(f"df/dx: autograd={x.grad.item()}, manual={...}")
# print(f"df/dy: autograd={y.grad.item()}, manual={...}")
# print(f"df/dz: autograd={z.grad.item()}, manual={...}")

### Exercise 2: Gradient Descent on a Simple Function

Use autograd to find the minimum of $f(x) = (x - 3)^2 + 1$ starting from $x = 10$.

Run 100 steps of gradient descent with learning rate $\eta = 0.1$.

In [ ]:
# TODO: Implement gradient descent using autograd
# x = torch.tensor(10.0, requires_grad=True)
# lr = 0.1

# for step in range(100):
#     ...

# print(f"Final x = {x.item():.4f} (should be close to 3.0)")

### Solutions

In [ ]:
# Solution 1: Multi-variable gradients
x = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)
z = torch.tensor(3.0, requires_grad=True)

f = x**2 * y + y * z**3 - 2 * x * z
f.backward()

# Manual gradients at (1, 2, 3)
manual_dx = 2 * 1.0 * 2.0 - 2 * 3.0     # 2xy - 2z = 4 - 6 = -2
manual_dy = 1.0**2 + 3.0**3              # x^2 + z^3 = 1 + 27 = 28
manual_dz = 3 * 2.0 * 3.0**2 - 2 * 1.0  # 3yz^2 - 2x = 54 - 2 = 52

print(f"f(1,2,3) = {f.item()}")
print(f"df/dx: autograd={x.grad.item():.1f}, manual={manual_dx:.1f}")
print(f"df/dy: autograd={y.grad.item():.1f}, manual={manual_dy:.1f}")
print(f"df/dz: autograd={z.grad.item():.1f}, manual={manual_dz:.1f}")

In [ ]:
# Solution 2: Gradient descent
x = torch.tensor(10.0, requires_grad=True)
lr = 0.1
history = []

for step in range(100):
    # Zero gradients
    if x.grad is not None:
        x.grad.zero_()
    
    # Forward
    f = (x - 3) ** 2 + 1
    
    # Backward
    f.backward()
    
    # Record history
    history.append((x.item(), f.item()))
    
    # Update (without tracking)
    with torch.no_grad():
        x -= lr * x.grad

print(f"Step  0: x={history[0][0]:.4f}, f(x)={history[0][1]:.4f}")
print(f"Step 10: x={history[10][0]:.4f}, f(x)={history[10][1]:.4f}")
print(f"Step 50: x={history[50][0]:.4f}, f(x)={history[50][1]:.4f}")
print(f"Step 99: x={history[99][0]:.4f}, f(x)={history[99][1]:.4f}")
print(f"\nFinal x = {x.item():.6f} (target: 3.0)")

## 10. Common Mistakes & Debugging Tips

1. **Forgetting `zero_grad()`**: Gradients accumulate by default. Always zero them before each backward pass in a training loop.
   ```python
   optimizer.zero_grad()  # or param.grad.zero_()
   loss.backward()
   ```

2. **Calling `.backward()` on non-scalar**: If the output is not a scalar, you must pass a `gradient` argument or call `.sum()` first.
   ```python
   y = model(x)       # shape (N,)
   y.sum().backward()  # or y.backward(torch.ones_like(y))
   ```

3. **In-place operations on tensors that require grad**: In-place ops (e.g., `x += 1`) can break the computation graph.
   ```python
   x = x + 1   # OK (creates new tensor)
   x += 1      # may cause errors if x requires_grad
   ```

4. **"RuntimeError: Trying to backward through the graph a second time"**: The graph is freed after `backward()`. Use `retain_graph=True` if you need multiple backward passes.

5. **`.grad` is `None`**: This means `backward()` hasn't been called yet, or the tensor is not a leaf, or `requires_grad=False`.

6. **Modifying `.grad` accidentally**: Never modify `.grad` tensors outside of `zero_grad()`. Clone them if you need to manipulate gradient values.

7. **Memory leak from storing tensors with grad history**: If you accumulate loss values for logging, use `.item()` or `.detach()` to avoid holding the computation graph in memory.
   ```python
   # Bad: stores entire computation graph
   losses.append(loss)
   # Good: stores just the number
   losses.append(loss.item())
   ```

8. **Debugging gradients**: Use `torch.autograd.gradcheck()` to numerically verify your custom autograd functions:
   ```python
   from torch.autograd import gradcheck
   input = torch.randn(3, requires_grad=True, dtype=torch.float64)
   gradcheck(my_function, input)  # returns True if correct
   ```